# Chapter 11: Evaluating voice agents

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [Evaluating voice agents](https://books.sugiv.fyi/voice_applications/evaluating-voice-agents.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## What to measure

## Sixteen calls is a wide interval

In [ ]:
import glob
import json
import re
import sys
from collections import Counter

import numpy as np

sys.path.insert(0, "sim")
src = open("building-the-receptionist.md").read()  # Chapter 8's judge, read from that chapter so they can't drift
fence = "`" * 3  # written this way so this cell can contain it
cells = re.findall(fence + r"\{code-cell\} ipython3\n(.*?)" + fence, src, re.S)
ns = {}
exec(cells[0].split("print(json.dumps")[0], ns)
exec([c for c in cells if "def judge(" in c][0].split("RUNS = [")[0], ns)
judge, BY_ID = ns["judge"], ns["BY_ID"]


def wilson(k, n, z=1.96):
    """95% interval for a proportion k/n that behaves well for small n."""
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half


for k, n in [(11, 16), (110, 160), (1100, 1600)]:
    lo, hi = wilson(k, n)
    print(f"{k:5d} of {n:5d} handled: {k / n:.0%}, 95% interval {lo:.0%} to {hi:.0%}")

## The same calls, different callers

In [ ]:
def calls(path):
    return json.load(open(path))["calls"]


conditions = {c: calls(f"data/ch11/talk_{c}.json") for c in ["base", "noisy", "voices", "both"]}
scores = {c: {cid: judge(BY_ID[cid], t) for cid, t in log.items()} for c, log in conditions.items()}
print("condition  handled  wrong actions")
for c, s in scores.items():
    print(f"{c:10s} {sum(x['handled'] for x in s.values()):4d}/16 {sum(x['wrong'] for x in s.values()):10d}")
cids = sorted(BY_ID)
print("\n      base noisy voices both")
for cid in cids:
    marks = ["  ok " if scores[c][cid]["handled"] else "  -- " for c in scores]
    if len(set(marks)) > 1:
        print(cid, "".join(marks), BY_ID[cid]["kind"])
pooled = sum(x["handled"] for s in scores.values() for x in s.values())
lo, hi = wilson(pooled, 64)
print(f"\npooled: {pooled} of 64, 95% interval {lo:.0%} to {hi:.0%}")

## The same calls, run again

In [ ]:
sampled = {s: {cid: judge(BY_ID[cid], t) for cid, t in calls(f"data/ch11/talk_base-sampled-{s}.json").items()}
           for s in range(5)}
totals = [sum(x["handled"] for x in r.values()) for r in sampled.values()]
print("handled in five sampled runs:", totals)
print("\ncalls that changed between runs:")
for cid in cids:
    marks = "".join("ok " if sampled[s][cid]["handled"] else "-- " for s in sampled)
    if len(set(marks.split())) > 1:
        print(f"   {cid}  {marks}  {BY_ID[cid]['kind']}")

## Who decides "handled"?

In [ ]:
def kappa(a, b):
    """Agreement beyond chance between two yes/no raters."""
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    po = np.mean(a == b)
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return (po - pe) / (1 - pe)


judge1 = json.load(open("data/ch11/llm_judge.json"))
judge2 = json.load(open("data/ch11/llm_judge2.json"))
rows = []
for f in sorted(glob.glob("data/ch8/talk_*.json")) + sorted(glob.glob("data/ch11/talk_*.json")):
    run = ("ch8/" if "/ch8/" in f else "ch11/") + f.split("talk_")[1][:-5]
    for cid, turns in calls(f).items():
        rule = judge(BY_ID[cid], turns)
        rows.append({"call": cid, "rules": rule["handled"], "rules_invented": rule["invented"],
                     "judge1": judge1["calls"][f"{run}/{cid}"]["done"] > 0.5,
                     "judge2": judge2["calls"][f"{run}/{cid}"]["correct"] > 0.5,
                     "judge2_invented": judge2["calls"][f"{run}/{cid}"]["invented"] > 0.5})
rules = [r["rules"] for r in rows]
for name in ["judge1", "judge2"]:
    j = [r[name] for r in rows]
    print(f"{name}: agrees with the rules on {np.mean(np.array(j) == np.array(rules)):.0%} of {len(rows)}, "
          f"kappa {kappa(j, rules):+.2f}, says handled {np.mean(j):.0%} (rules: {np.mean(rules):.0%})")
print("\nfirst judge, where it disagreed with the rules:")
print("   rules say handled, judge says not:", Counter(r["call"] for r in rows if r["rules"] and not r["judge1"]).most_common(4))
print("   rules say failed, judge says handled:", Counter(r["call"] for r in rows if not r["rules"] and r["judge1"]).most_common(5))

In [ ]:
print("second judge, where it disagreed with the rules:")
print("   rules say handled, judge says not:", Counter(r["call"] for r in rows if r["rules"] and not r["judge2"]).most_common(4))
print("   rules say failed, judge says handled:", Counter(r["call"] for r in rows if not r["rules"] and r["judge2"]).most_common(5))
print("\nmade-up facts flagged: rules", sum(r["rules_invented"] for r in rows), ", second judge",
      sum(r["judge2_invented"] for r in rows), Counter(r["call"] for r in rows if r["judge2_invented"]).most_common(3))

## Rerun the models

Everything above used the saved results. The cells below rerun the models that produced them, on a GPU (in Colab:
Runtime, then Change runtime type, then T4 GPU). Each step builds the software it needs in its own environment the
first time, which takes a few minutes, then runs. The first cell below sets up a two-item check that everything
works, on a scratch copy of the folder, so the saved results stay as they are. Delete its lines for a full rerun,
which overwrites the saved results with yours. On GPUs without bfloat16, such as the T4, the scripts use float16
automatically.

In [ ]:
# A two-item check that every step runs, on a scratch copy so the saved results stay untouched.
# Delete these lines for a full rerun, which overwrites the saved results with your own.
import os, shutil, tempfile
os.environ["VOTF_QUICK"] = "2"
scratch = os.path.join(tempfile.gettempdir(), "votf-quick")
shutil.copytree(".", scratch, dirs_exist_ok=True)
os.chdir(scratch)

In [ ]:
!bash tools/stack.sh tts
!~/.votf/envs/tts/bin/python sim/capture_ch11.py --part voice

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch11.py --part hear

In [ ]:
!bash tools/stack.sh hf
!~/.votf/envs/hf/bin/python sim/capture_ch11.py --part talk

The judge is Qwen3-8B, which needs more than a T4's 16 GB.

In [ ]:
!bash tools/stack.sh hf
!~/.votf/envs/hf/bin/python sim/capture_ch11.py --part judge

The same 8B judge, given the expected behaviour.

In [ ]:
!bash tools/stack.sh hf
!~/.votf/envs/hf/bin/python sim/capture_ch11.py --part judge2